# Лабораторная 1.1. Выбор и загрузка корпуса

Цель: загрузить не менее 5000 документов из открытого корпуса и описать их.

Корпус: Russian Wikipedia subset.  
Источник: Hugging Face datasets, `wikimedia/wikipedia`, конфигурация `20231101.ru`.

In [2]:
import json
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
from datasets import load_dataset
from tqdm.auto import tqdm

In [3]:
DATASET_ID = "wikimedia/wikipedia"
CONFIG = "20231101.ru"
SPLIT = "train"

TARGET_DOCS = 5000
MIN_CHARS = 300


def find_project_root(start: Path) -> Path:
    """Поднимаемся вверх по папкам, пока не найдём project_state.md."""
    current = start.resolve()
    for folder in [current, *current.parents]:
        if (folder / "project_state.md").exists():
            return folder
    raise FileNotFoundError("Не удалось найти project_state.md, старт: " + str(start))


PROJECT_ROOT = find_project_root(Path.cwd())
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Raw data dir:", RAW_DIR)
print("Processed data dir:", PROCESSED_DIR)

Project root: C:\Projects\ai-playground\Project1
Raw data dir: C:\Projects\ai-playground\Project1\data\raw
Processed data dir: C:\Projects\ai-playground\Project1\data\processed


In [4]:
preview_dataset = load_dataset(
    DATASET_ID,
    CONFIG,
    split=SPLIT,
    streaming=True
)

first_doc = next(iter(preview_dataset))

print("Keys:", first_doc.keys())
print("ID:", first_doc.get("id"))
print("Title:", first_doc.get("title"))
print("URL:", first_doc.get("url"))
print("Text preview:")
print(str(first_doc.get("text"))[:500])

Resolving data files:   0%|          | 0/21 [00:00<?, ?it/s]

Keys: dict_keys(['id', 'url', 'title', 'text'])
ID: 7
Title: Литва
URL: https://ru.wikipedia.org/wiki/%D0%9B%D0%B8%D1%82%D0%B2%D0%B0
Text preview:
Литва́ ( ), официальное название — Лито́вская Респу́блика () — государство, расположенное в Северной Европе. Площадь —  км². Протяжённость с севера на юг — 280 км, а с запада на восток — 370 км. Население составляет  человек (август, 2023). Занимает 137-е место в мире по численности населения и 121-е по территории. Имеет выход к Балтийскому морю, расположена на его восточном побережье. Береговая линия составляет всего 99 км (наименьший показатель среди государств Балтии). На севере граничит с Ла


In [5]:
from tqdm import tqdm

stream = load_dataset(
    DATASET_ID,
    CONFIG,
    split=SPLIT,
    streaming=True
)

records = []
seen_ids = set()
seen_titles = set()

progress = tqdm(total=TARGET_DOCS, desc="Collecting docs")

for row in stream:
    doc_id = str(row.get("id") or "").strip()
    title = str(row.get("title") or "").strip()
    text = str(row.get("text") or "").strip()
    source = str(row.get("url") or "").strip()

    if not doc_id:
        doc_id = f"wiki_{len(records):06d}"

    if not title:
        continue

    if len(text) < MIN_CHARS:
        continue

    if doc_id in seen_ids:
        continue

    if title.lower() in seen_titles:
        continue

    seen_ids.add(doc_id)
    seen_titles.add(title.lower())

    records.append({
        "doc_id": doc_id,
        "title": title,
        "text": text,
        "source": source,
        "date": "",
        "metadata": {
            "dataset": DATASET_ID,
            "config": CONFIG,
            "language": "ru",
            "loaded_at": datetime.now(timezone.utc).isoformat(),
        },
    })

    progress.update(1)

    if len(records) >= TARGET_DOCS:
        break

progress.close()

df = pd.DataFrame(records)

print("Shape:", df.shape)
print(df[["doc_id", "title"]].head(10))

Resolving data files:   0%|          | 0/21 [00:00<?, ?it/s]

Shape: (5000, 6)
  doc_id          title
0      7          Литва
1      9         Россия
2     10       Слоновые
3     11        Мамонты
4     15  Красная книга
5     16      Соционика
6     18          Школа
7     20    Лингвистика
8     21     Социология
9     27  Киевская Русь


In [6]:
from pathlib import Path

print("Current dir:", Path.cwd())
print("project_state.md here?", (Path.cwd() / "project_state.md").exists())

Current dir: c:\Projects\ai-playground\Project1\notebooks
project_state.md here? False


In [7]:
PROJECT_ROOT = Path.cwd().parent  # поднимаемся из notebooks/ в корень проекта

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print(RAW_DIR.resolve())

C:\Projects\ai-playground\Project1\data\raw


In [8]:
for i in range(3):
    row = df.iloc[i]
    print("=" * 80)
    print("doc_id:", row["doc_id"])
    print("title:", row["title"])
    print("source:", row["source"])
    print("chars:", len(row["text"]))
    print(row["text"][:300].replace("\n", " "))
    print()

doc_id: 7
title: Литва
source: https://ru.wikipedia.org/wiki/%D0%9B%D0%B8%D1%82%D0%B2%D0%B0
chars: 43240
Литва́ ( ), официальное название — Лито́вская Респу́блика () — государство, расположенное в Северной Европе. Площадь —  км². Протяжённость с севера на юг — 280 км, а с запада на восток — 370 км. Население составляет  человек (август, 2023). Занимает 137-е место в мире по численности населения и 121-

doc_id: 9
title: Россия
source: https://ru.wikipedia.org/wiki/%D0%A0%D0%BE%D1%81%D1%81%D0%B8%D1%8F
chars: 156427
Росси́я, или Росси́йская Федера́ция (), — государство в Восточной Европе и Северной Азии. Россия — крупнейшее государство в мире, её территория в международно признанных границах составляет  км². Население страны в тех же границах, но включая территорию украинского Крыма, аннексия Россией которого н

doc_id: 10
title: Слоновые
source: https://ru.wikipedia.org/wiki/%D0%A1%D0%BB%D0%BE%D0%BD%D0%BE%D0%B2%D1%8B%D0%B5
chars: 20798
Слоно́вые, или слоны́ , — семейство класса млекопит

In [9]:
df["chars"] = df["text"].str.len()
df["words"] = df["text"].str.split().str.len()

stats = {
    "lab": "1.1",
    "corpus": "Russian Wikipedia subset",
    "dataset": DATASET_ID,
    "config": CONFIG,
    "documents": int(len(df)),
    "avg_chars": round(float(df["chars"].mean()), 1),
    "median_chars": float(df["chars"].median()),
    "min_chars": int(df["chars"].min()),
    "max_chars": int(df["chars"].max()),
    "avg_words": round(float(df["words"].mean()), 1),
    "median_words": float(df["words"].median()),
    "min_words": int(df["words"].min()),
    "max_words": int(df["words"].max()),
}

print(json.dumps(stats, ensure_ascii=False, indent=2))
print()
print(df[["chars", "words"]].describe())

{
  "lab": "1.1",
  "corpus": "Russian Wikipedia subset",
  "dataset": "wikimedia/wikipedia",
  "config": "20231101.ru",
  "documents": 5000,
  "avg_chars": 12406.4,
  "median_chars": 4405.0,
  "min_chars": 300,
  "max_chars": 182051,
  "avg_words": 1655.4,
  "median_words": 586.0,
  "min_words": 25,
  "max_words": 24503
}

              chars        words
count    5000.00000   5000.00000
mean    12406.36320   1655.38160
std     19467.36339   2598.92819
min       300.00000     25.00000
25%      1277.75000    172.00000
50%      4405.00000    586.00000
75%     15854.25000   2112.50000
max    182051.00000  24503.00000


In [10]:
out_data = RAW_DIR / "ru_wikipedia_5000.jsonl"
out_stats = PROCESSED_DIR / "lab1_1_corpus_stats.json"

save_df = df.drop(columns=["chars", "words"])
save_df.to_json(out_data, orient="records", lines=True, force_ascii=False)

out_stats.write_text(
    json.dumps(stats, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

size_mb = out_data.stat().st_size / (1024 * 1024)
print("Corpus:", out_data.resolve())
print("Stats:", out_stats.resolve())
print(f"Size: {size_mb:.1f} MB")

Corpus: C:\Projects\ai-playground\Project1\data\raw\ru_wikipedia_5000.jsonl
Stats: C:\Projects\ai-playground\Project1\data\processed\lab1_1_corpus_stats.json
Size: 108.4 MB


In [11]:
check = pd.read_json(out_data, lines=True)

print("Rows:", len(check))
print("Columns:", check.columns.tolist())
print("First title:", check.iloc[0]["title"])
print(check.iloc[0]["text"][:200])

Rows: 5000
Columns: ['doc_id', 'title', 'text', 'source', 'date', 'metadata']
First title: Литва
Литва́ ( ), официальное название — Лито́вская Респу́блика () — государство, расположенное в Северной Европе. Площадь —  км². Протяжённость с севера на юг — 280 км, а с запада на восток — 370 км. Насел


## Вывод по Лаб 1.1

Корпус: подмножество русской Википедии, Hugging Face `wikimedia/wikipedia`, config `20231101.ru`.

Загружено документов: 5000.
Схема записи: doc_id, title, text, source, date, metadata.

Средний объём документа: ~12400 символов (~1655 слов).
Медианный объём: 4405 символов (586 слов).
Разброс: от 300 до 182051 символов.

Хранение: JSON Lines, `data/raw/ru_wikipedia_5000.jsonl`.
Статистика: `data/processed/lab1_1_corpus_stats.json`.

Ограничения: срез взят последовательно (не случайная выборка);
поле `date` пустое, так как в снапшоте нет дат публикации статей.